# ACTO SDK: Async Proof Operations

This notebook demonstrates async/await support for proof creation and verification.

## Installation

```bash
pip install actobotics
```

> **Note:** The package is called `actobotics` on PyPI, but you import it as `acto` in Python.


In [ ]:
import asyncio
from datetime import datetime

from acto.crypto import KeyPair
from acto.proof import create_proof_async
from acto.client import AsyncACTOClient
from acto.telemetry.models import TelemetryBundle, TelemetryEvent

# API credentials - get from https://https://acto-production.up.railway.app/dashboard
API_KEY = "YOUR_API_KEY"
WALLET_ADDRESS = "YOUR_WALLET_ADDRESS"


## Async Proof Creation


In [ ]:
async def create_proof_example():
    # Generate keypair
    keypair = KeyPair.generate()
    
    # Create telemetry bundle
    bundle = TelemetryBundle(
        task_id="async-task-001",
        robot_id="robot-001",
        events=[
            TelemetryEvent(
                ts=datetime.now().isoformat(),
                topic="imu",
                data={"accel": [1.0, 2.0, 3.0]}
            )
        ]
    )
    
    # Create proof asynchronously
    envelope = await create_proof_async(
        bundle,
        keypair.private_key_b64,
        keypair.public_key_b64
    )
    
    print(f"Proof created: {envelope.payload.payload_hash}")
    return envelope

# Run async function
envelope = await create_proof_example()


## Async Proof Verification via API

Proof verification must be done through the ACTO API using the async client.


In [ ]:
async def verify_proof_example():
    # Initialize async API client
    client = AsyncACTOClient(
        api_key=API_KEY,
        wallet_address=WALLET_ADDRESS
    )
    
    # Verify proof via API
    try:
        result = await client.verify(envelope)
        print(f"Proof is valid: {result.valid}")
        print(f"Reason: {result.reason}")
    except Exception as e:
        print(f"Verification failed: {e}")
        print("Make sure to set API_KEY and WALLET_ADDRESS!")

await verify_proof_example()


## Batch Async Operations


In [ ]:
async def batch_create_proofs():
    keypair = KeyPair.generate()
    
    # Create multiple proofs concurrently
    tasks = []
    for i in range(5):
        bundle = TelemetryBundle(
            task_id=f"task-{i:03d}",
            robot_id="robot-001",
            events=[
                TelemetryEvent(
                    ts=datetime.now().isoformat(),
                    topic="sensor",
                    data={"data": i}
                )
            ]
        )
        tasks.append(
            create_proof_async(
                bundle,
                keypair.private_key_b64,
                keypair.public_key_b64
            )
        )
    
    # Wait for all proofs to be created
    envelopes = await asyncio.gather(*tasks)
    print(f"Created {len(envelopes)} proofs")
    
    # Batch verify all proofs via API
    client = AsyncACTOClient(api_key=API_KEY, wallet_address=WALLET_ADDRESS)
    try:
        result = await client.verify_batch(envelopes)
        print(f"Verified: {result.valid_count}/{result.total} proofs valid")
    except Exception as e:
        print(f"Batch verification failed: {e}")
        print("Make sure to set API_KEY and WALLET_ADDRESS!")

await batch_create_proofs()
